## 04 Modeling

Train and tune models here.

In [1]:
%pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.2.2
[notice] To update, run: C:\Users\ADMIN\env_2026\Scripts\python.exe -m pip install --upgrade pip


### Nhiệm vụ #44 – Kiểm tra Overfitting và Data Leakage
 

#### Xác định dữ liệu và mục tiêu dự đoán

In [2]:

from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "games_cleaned.csv"

df = pd.read_csv(DATA_PATH, low_memory=False)

print("Kích thước dữ liệu:", df.shape)
print("\nDanh sách các cột:")
print(df.columns.tolist())

print("\nNăm dòng dữ liệu đầu tiên:")
display(df.head())


Kích thước dữ liệu: (125846, 21)

Danh sách các cột:
['id', 'Name', 'Release date', 'Estimated owners', 'Peak CCU', 'Price', 'Discount', 'DLC count', 'Positive', 'Negative', 'User score', 'Metacritic score', 'Recommendations', 'Achievements', 'Average playtime forever', 'Average playtime two weeks', 'Median playtime forever', 'Median playtime two weeks', 'Categories', 'Genres', 'Tags']

Năm dòng dữ liệu đầu tiên:


,id,Name,Release date,Estimated owners,Peak CCU,Price,Discount,DLC count,Positive,Negative,...,Metacritic score,Recommendations,Achievements,Average playtime forever,Average playtime two weeks,Median playtime forever,Median playtime two weeks,Categories,Genres,Tags
0,10,Counter-Strike,2000-11-01,10000000 - 20000000,7323,1.99,80,0,243818,6427,...,88.0,164682,0,10735,836,185,99,"Multi-player,PvP,Online PvP,Shared/Split Scree...",Action,"Action,FPS,Multiplayer,Shooter,Classic,Team-Ba..."
1,20,Team Fortress Classic,1999-04-01,1000000 - 2000000,66,1.24,75,0,7602,1136,...,NaN,6742,0,466,0,13,0,"Multi-player,PvP,Online PvP,Shared/Split Scree...",Action,"Action,FPS,Multiplayer,Classic,Hero Shooter,Sh..."
2,30,Day of Defeat,2003-05-01,5000000 - 10000000,87,1.24,75,0,6414,688,...,79.0,4351,0,997,948,17,948,"Multi-player,Camera Comfort,Color Alternatives...",Action,"FPS,World War II,Multiplayer,Shooter,Action,Wa..."
3,40,Deathmatch Classic,2001-06-01,5000000 - 10000000,7,1.24,75,0,2618,545,...,NaN,2360,0,339,0,7,0,"Multi-player,PvP,Online PvP,Shared/Split Scree...",Action,"Action,FPS,Classic,Multiplayer,Shooter,First-P..."
4,50,Half-Life: Opposing Force,1999-11-01,2000000 - 5000000,74,1.24,75,0,24363,1198,...,NaN,23464,0,415,2,202,2,"Single-player,Multi-player,Custom Volume Contr...",Action,"FPS,Action,Classic,Sci-fi,Singleplayer,Shooter..."


#### Chọn mục tiêu dự đoán
Mục tiêu dự đoán	Ý nghĩa	Lưu ý
Peak CCU	Lượng người chơi đồng thời cao nhất	Phù hợp để thử dự đoán mức độ thu hút người chơi
Positive	Số lượt đánh giá tích cực	Có thể chịu ảnh hưởng lớn từ quy mô người chơi
Price	Giá game	Có thể thử dự đoán giá dựa trên các thuộc tính của game
→ Chọn Peak CCU trước.

#### Kiểm tra dữ liệu trước khi xây dựng mô hình

In [3]:
# Kiểm tra các cột có thể dùng để dự đoán Peak CCU

target = "Peak CCU"

candidate_features = [
    "Price",
    "Discount",
    "DLC count",
    "Achievements",
    "Metacritic score",
    "User score",
    "Positive",
    "Negative",
    "Recommendations",
    "Average playtime forever",
    "Average playtime two weeks",
    "Median playtime forever",
    "Median playtime two weeks",
]

print("Mục tiêu dự đoán:", target)

print("\nKiểu dữ liệu của các cột:")
display(df[[target] + candidate_features].dtypes.to_frame("Data type"))

print("\nSố lượng giá trị thiếu:")
display(df[[target] + candidate_features].isna().sum().to_frame("Missing values"))

print("\nThống kê mô tả:")
display(df[[target] + candidate_features].describe().T)

Mục tiêu dự đoán: Peak CCU

Kiểu dữ liệu của các cột:


,Data type
Peak CCU,int64
Price,float64
Discount,int64
DLC count,int64
Achievements,int64
Metacritic score,float64
User score,float64
Positive,int64
Negative,int64
Recommendations,int64



Số lượng giá trị thiếu:


,Missing values
Peak CCU,0
Price,0
Discount,0
DLC count,0
Achievements,0
Metacritic score,121588
User score,125806
Positive,0
Negative,0
Recommendations,0



Thống kê mô tả:


,count,mean,std,min,25%,50%,75%,max
Peak CCU,125846.0,53.191003,3681.214565,0.0,0.00,0.00,0.00,1013936.00
Price,125846.0,4.810724,12.489443,0.0,0.59,2.39,5.52,999.98
Discount,125846.0,17.882634,28.621900,0.0,0.00,0.00,36.00,100.00
DLC count,125846.0,0.535472,14.328929,0.0,0.00,0.00,0.00,3703.00
Achievements,125846.0,17.923184,139.701333,0.0,0.00,2.00,19.00,9821.00
Metacritic score,4258.0,73.895726,10.262581,6.0,68.00,76.00,81.00,97.00
User score,40.0,75.250000,17.642824,37.0,62.50,77.00,89.00,100.00
Positive,125846.0,1018.254907,27728.814551,0.0,0.00,4.00,35.00,7642084.00
Negative,125846.0,164.890422,5305.190176,0.0,0.00,1.00,10.00,1173003.00
Recommendations,125846.0,937.360401,21595.033672,0.0,0.00,0.00,0.00,4830455.00


#### Tạo mô hình thử nghiệm để kiểm tra overfitting
Chia dữ liệu thành tập huấn luyện và tập kiểm tra.



Dùng log1p(Peak CCU) để giảm ảnh hưởng của các giá trị CCU cực lớn.



So sánh mô hình đơn giản với Random Forest.



So sánh sai số giữa train và test để phát hiện dấu hiệu overfitting.

In [4]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, r2_score

# Chọn các đặc trưng cơ bản, tránh dùng số liệu đánh giá
# (giá, giảm giá, số DLC, số thành tựu: các đặc điểm của game liên quan RQ1 và RQ5)
features = [
    "Price",
    "Discount",
    "DLC count",
    "Achievements"
]

target = "Peak CCU"

# Chuyển các cột đầu vào về dạng số nếu có thể
X = df[features].copy()

for col in X.columns:
    X[col] = pd.to_numeric(X[col], errors="coerce")

y = np.log1p(df[target])

# Chia train/test trước khi huấn luyện
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

models = {
    "Baseline": make_pipeline(
        SimpleImputer(strategy="median"),
        DummyRegressor(strategy="median")
    ),
    "Random Forest": make_pipeline(
        SimpleImputer(strategy="median"),
        RandomForestRegressor(
            n_estimators=100,
            max_depth=10,
            random_state=42,
            n_jobs=-1
        )
    )
}

results = []

for name, model in models.items():
    model.fit(X_train, y_train)

    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test)

    results.append({
        "Model": name,
        "Train MAE": mean_absolute_error(y_train, train_pred),
        "Test MAE": mean_absolute_error(y_test, test_pred),
        "Train R2": r2_score(y_train, train_pred),
        "Test R2": r2_score(y_test, test_pred)
    })

results_df = pd.DataFrame(results)
display(results_df.round(4))

,Model,Train MAE,Test MAE,Train R2,Test R2
0,Baseline,0.3251,0.3234,-0.1002,-0.0988
1,Random Forest,0.3703,0.3954,0.4282,0.3248


Cách đọc kết quả
- Train MAE thấp nhưng Test MAE cao hơn nhiều: mô hình có dấu hiệu overfitting.
- Train và Test MAE đều cao: mô hình có thể chưa học được quy luật hữu ích.
- Random Forest tốt hơn Baseline trên tập test: có dấu hiệu mô hình học được thông tin hữu ích từ các đặc trưng đã chọn.
- Test R² âm: mô hình dự đoán còn kém hơn một phương án đơn giản dự đoán bằng giá trị trung bình của tập test.

#### Kiểm tra lại đặc điểm dữ liệu
Kết quả này giúp xác định vì sao mô hình khó dự đoán: dữ liệu có thể tập trung quá nhiều ở mức CCU bằng 0, trong khi một số ít game có giá trị cực lớn.

In [5]:
print("Tỷ lệ Peak CCU bằng 0:")
print((df["Peak CCU"] == 0).mean() * 100, "%")

print("\nSố game có Peak CCU > 0:")
print((df["Peak CCU"] > 0).sum())

print("\nCác phân vị của Peak CCU:")
display(df["Peak CCU"].quantile([0, 0.5, 0.75, 0.9, 0.95, 0.99, 1]))

Tỷ lệ Peak CCU bằng 0:
84.3618390731529 %

Số game có Peak CCU > 0:
19680

Các phân vị của Peak CCU:


0.00          0.00
0.50          0.00
0.75          0.00
0.90          2.00
0.95          8.00
0.99        215.55
1.00    1013936.00
Name: Peak CCU, dtype: float64


Kết quả này xác nhận một vấn đề quan trọng: dữ liệu Peak CCU bị lệch rất mạnh về phía 0. Đây có thể là một trong những nguyên nhân khiến mô hình khó dự đoán chính xác lượng người chơi.

In [6]:
#Ý nghĩa nhãn: 0 là game có Peak CCU = 0; 1 là game có Peak CCU > 0.

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    f1_score
)

# Chọn đặc trưng và tạo nhãn phân loại
features = ["Price", "Discount", "DLC count", "Achievements"]

X = df[features].copy()
y = (df["Peak CCU"] > 0).astype(int)

# Chuyển dữ liệu đầu vào về dạng số
for col in X.columns:
    X[col] = pd.to_numeric(X[col], errors="coerce")

# Chia tập train và test, giữ tỷ lệ hai nhóm
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Xây dựng mô hình phân loại
model_stage1 = make_pipeline(
    SimpleImputer(strategy="median"),
    RandomForestClassifier(
        n_estimators=100,
        max_depth=10,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    )
)

model_stage1.fit(X_train, y_train)

# Dự đoán trên train và test
train_pred = model_stage1.predict(X_train)
test_pred = model_stage1.predict(X_test)

# Đánh giá kết quả
print("Accuracy train:", round(accuracy_score(y_train, train_pred), 4))
print("Accuracy test:", round(accuracy_score(y_test, test_pred), 4))

print("\nF1-score test:", round(f1_score(y_test, test_pred), 4))

print("\nClassification report trên tập test:")
print(classification_report(
    y_test, test_pred,
    target_names=["Peak CCU = 0", "Peak CCU > 0"],
    zero_division=0
))

print("Confusion matrix trên tập test:")
display(pd.DataFrame(
    confusion_matrix(y_test, test_pred),
    index=["Thực tế = 0", "Thực tế > 0"],
    columns=["Dự đoán = 0", "Dự đoán > 0"]
))

Accuracy train: 0.7784
Accuracy test: 0.7688

F1-score test: 0.4828

Classification report trên tập test:
              precision    recall  f1-score   support

Peak CCU = 0       0.93      0.78      0.85     21234
Peak CCU > 0       0.37      0.69      0.48      3936

    accuracy                           0.77     25170
   macro avg       0.65      0.74      0.67     25170
weighted avg       0.84      0.77      0.79     25170

Confusion matrix trên tập test:


,Dự đoán = 0,Dự đoán > 0
Thực tế = 0,16635,4599
Thực tế > 0,1220,2716


Kết quả giai đoạn 1 cho thấy mô hình phát hiện được một phần game có Peak CCU > 0, nhưng vẫn dự đoán nhầm khá nhiều.

#### Giai đoạn 2: dự đoán lượng Peak CCU của các game có CCU lớn hơn 0
Lưu ý: Giai đoạn 2 chỉ đánh giá hồi quy trên các game thực sự có Peak CCU > 0. Kết quả này chưa đại diện cho hiệu quả toàn bộ mô hình hai giai đoạn, vì chưa tính các lỗi phân loại từ giai đoạn 1.

In [7]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
import numpy as np
import pandas as pd

# Chọn các game có Peak CCU > 0
positive_mask = df["Peak CCU"] > 0

X_positive = df.loc[positive_mask, features].copy()
y_positive = np.log1p(df.loc[positive_mask, "Peak CCU"])

# Chuyển các đặc trưng về dạng số
for col in X_positive.columns:
    X_positive[col] = pd.to_numeric(X_positive[col], errors="coerce")

# Chia dữ liệu theo đúng các chỉ số train/test đã dùng ở giai đoạn 1
train_index = X_train.index.intersection(X_positive.index)
test_index = X_test.index.intersection(X_positive.index)

X_train_stage2 = X_positive.loc[train_index]
y_train_stage2 = y_positive.loc[train_index]

X_test_stage2 = X_positive.loc[test_index]
y_test_stage2 = y_positive.loc[test_index]

# Tạo và huấn luyện mô hình hồi quy
model_stage2 = make_pipeline(
    SimpleImputer(strategy="median"),
    RandomForestRegressor(
        n_estimators=100,
        max_depth=10,
        random_state=42,
        n_jobs=-1
    )
)

model_stage2.fit(X_train_stage2, y_train_stage2)

# Dự đoán trên train và test
train_pred_stage2 = model_stage2.predict(X_train_stage2)
test_pred_stage2 = model_stage2.predict(X_test_stage2)

# Đánh giá trên thang log
print("Số game CCU > 0 trong train:", len(X_train_stage2))
print("Số game CCU > 0 trong test:", len(X_test_stage2))

print("\nTrain MAE:", round(
    mean_absolute_error(y_train_stage2, train_pred_stage2), 4
))
print("Test MAE:", round(
    mean_absolute_error(y_test_stage2, test_pred_stage2), 4
))
print("Train R2:", round(
    r2_score(y_train_stage2, train_pred_stage2), 4
))
print("Test R2:", round(
    r2_score(y_test_stage2, test_pred_stage2), 4
))

Số game CCU > 0 trong train: 15744
Số game CCU > 0 trong test: 3936

Train MAE: 0.9858
Test MAE: 1.108
Train R2: 0.4383
Test R2: 0.2449


Đánh giá kết quả
Chỉ số	Train	Test
MAE (thang log)	0.9858	1.1080
R²	0.4383	0.2449

- Overfitting: MAE giữa train và test chênh 0.1222, R² chênh 0.1934. R² trên tập test thấp hơn khá nhiều so với tập train, nên có dấu hiệu overfitting nhẹ.
- Khả năng dự đoán: Test R² = 0.2449, nghĩa là mô hình giải thích được khoảng 24,49% biến thiên của mục tiêu trên thang log.
- Sai số: MAE được tính trên log1p(Peak CCU), không phải số người chơi trực tiếp.

#### Kết hợp hai giai đoạn
Cách tính:
- Nếu giai đoạn 1 dự đoán CCU = 0, dự đoán cuối cùng là 0.
- Nếu giai đoạn 1 dự đoán CCU > 0, lấy dự đoán hồi quy của giai đoạn 2 rồi chuyển về thang CCU bằng expm1.
- So sánh dự đoán cuối cùng với Peak CCU thực tế trên toàn bộ tập test.
**Lưu ý:** Đoạn code này giả định hai mô hình đã được huấn luyện bằng đúng các đặc trưng và cách chia dữ liệu hiện tại. Chỉ số MAE và R² cuối cùng được tính trên thang CCU gốc; do đó không so sánh trực tiếp MAE này với MAE trên thang log ở giai đoạn 2.

In [8]:
# Kết hợp dự đoán của hai giai đoạn trên tập test

stage1_pred = model_stage1.predict(X_test)

stage2_pred_log = model_stage2.predict(X_test)
stage2_pred_ccu = np.expm1(stage2_pred_log)

# Chỉ giữ dự đoán CCU dương nếu giai đoạn 1 phân loại là dương
final_pred = np.where(stage1_pred == 1, stage2_pred_ccu, 0)

# Giá trị CCU thực tế trên toàn bộ tập test
y_test_ccu = df.loc[X_test.index, "Peak CCU"].to_numpy()

print("MAE toàn bộ test:", round(
    mean_absolute_error(y_test_ccu, final_pred), 2
))
print("R2 toàn bộ test:", round(
    r2_score(y_test_ccu, final_pred), 4
))

MAE toàn bộ test: 49.1
R2 toàn bộ test: 0.0055


Kết luận nhiệm vụ # 44 đến hiện tại
- Overfitting: Giai đoạn 1 không có dấu hiệu rõ (accuracy train 0,778, test 0,769). Giai đoạn 2 có dấu hiệu overfitting nhẹ (R² train 0,438, test 0,245).
- Data leakage: Đã hạn chế một số nguy cơ bằng cách không dùng các cột đánh giá và đề xuất, nhưng chưa thể khẳng định đã loại bỏ hoàn toàn leakage.
- Hiệu quả mô hình: Mô hình hai giai đoạn có R² toàn bộ test chỉ 0,0055, nên khả năng dự đoán CCU thực tế còn yếu.
- Nguyên nhân cần xem xét: 84,36% game có CCU bằng 0, phân bố lệch mạnh và các đặc trưng hiện tại (Price, Discount, DLC count, Achievements) có thể chưa đủ thông tin để dự đoán mức độ phổ biến.